In [1]:
from pathlib import Path
import os
import sys
import importlib

PROJECT_ROOT = Path(
    "/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag"
)

# Set working directory
os.chdir(PROJECT_ROOT)

# Make local project modules importable
project_path = str(PROJECT_ROOT)

if project_path not in sys.path:
    sys.path.insert(0, project_path)

# Refresh Python's module discovery
importlib.invalidate_caches()

print("Project root:", PROJECT_ROOT)
print("Working directory:", os.getcwd())
print("Project on sys.path:", project_path in sys.path)

Project root: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag
Working directory: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag
Project on sys.path: True


In [2]:
from config import DATA_DIR
from graph.neo4j_client import get_driver
from graph.schema import NODE_SCHEMAS, RELATIONSHIP_SCHEMAS
from ingestion.load_documents import load_documents

In [3]:
from dotenv import load_dotenv
load_dotenv(".env", override=True)
print("Environment loaded.")
print("NEBIUS API key set:", bool(os.getenv("NEBIUS_API_KEY")))

print("Neo4j URI:", os.getenv("NEO4J_URI"))
print("Neo4j username:", os.getenv("NEO4J_USERNAME"))
print("Neo4j password set:", bool(os.getenv("NEO4J_PASSWORD")))

Environment loaded.
NEBIUS API key set: True
Neo4j URI: neo4j+s://2725a5d2.databases.neo4j.io
Neo4j username: neo4j
Neo4j password set: True


In [4]:
import graph

print("graph imported from:")
print(graph.__file__)

graph imported from:
/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/graph/__init__.py


In [5]:
from neo4j import GraphDatabase

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("✅ Neo4j connection successful")
driver.close()

✅ Neo4j connection successful


In [6]:
from graph.neo4j_client import get_driver

driver = get_driver()

driver.verify_connectivity()

print("✅ graph/neo4j_client.py works correctly")

driver.close()

✅ graph/neo4j_client.py works correctly


In [8]:
from openai import OpenAI
import os

nebius_client = OpenAI(
    api_key=os.getenv("NEBIUS_API_KEY"),
    base_url="https://api.studio.nebius.com/v1"
)

models = nebius_client.models.list()

for model in models.data[:20]:
    print(model.id)

meta-llama/Llama-3.3-70B-Instruct
Qwen/Qwen3-235B-A22B-Instruct-2507
Qwen/Qwen3-32B
google/gemma-3-27b-it
Qwen/Qwen2.5-VL-72B-Instruct
Qwen/Qwen3-Embedding-8B
openai/gpt-oss-120b
Qwen/Qwen3-30B-A3B-Instruct-2507
NousResearch/Hermes-4-70B
NousResearch/Hermes-4-405B
nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B
zai-org/GLM-5.1
nvidia/Cosmos3-Super-Reasoner
openbmb/MiniCPM-V-4_5
nvidia/Nemotron-3-Nano-Omni
Qwen/Qwen3-Next-80B-A3B-Thinking
nvidia/Llama-3_1-Nemotron-Ultra-253B-v1
nvidia/Nemotron-3_5-Lightning
deepseek-ai/DeepSeek-V4-Flash
MiniMaxAI/MiniMax-M3


In [ ]:
#Using Qwen/Qwen3-30B-A3B-Instruct-2507 due to token constraints. Did not receive Nebius credits until after the project's halfway point, so I was unable to test Nebius models. I will test them in the future and update the code accordingly.'
from llama_index.llms.openai_like import OpenAILike
import os

llm = OpenAILike(
    model="Qwen/Qwen3-30B-A3B-Instruct-2507",
    api_base="https://api.studio.nebius.com/v1",
    api_key=os.getenv("NEBIUS_API_KEY"),
    is_chat_model=True,
    is_function_calling_model=False,
)

response = llm.complete("Reply with exactly: Nebius works")

print(response)

Nebius works


In [ ]:
#Test the Nebius API with the OpenAI client and see if the model works
client = OpenAI(
    api_key=os.getenv("NEBIUS_API_KEY"),
    base_url="https://api.tokenfactory.nebius.com/v1"
)

response = client.chat.completions.create(
    model="Qwen/Qwen3-30B-A3B-Instruct-2507",
    messages=[
        {
            "role": "user",
            "content": "Return exactly the text: Qwen works"
        }
    ],
    max_tokens=20,
    temperature=0
)

print(response.choices[0].message.content)

Qwen works


In [13]:
%run evaluation/validate_dataset.py

Node counts: {'Team': 5, 'Skill': 8, 'Technology': 7, 'Person': 10, 'Project': 4, 'Decision': 8, 'Meeting': 6, 'Document': 33}
Total nodes: 81
Total relationships: 174
Evaluation questions: 10
Validation passed.


In [14]:
from ingestion.load_documents import load_documents

documents = load_documents()

print("Documents loaded:", len(documents))
doc = documents[0]

print(doc.text[:1000])
print(doc.metadata)

Documents loaded: 34
# Acme AI Synthetic Organizational Corpus

This fictional corpus is designed for a GraphRAG vs vector-RAG comparison.

Corpus folders:
- `people/`
- `projects/`
- `meetings/`
- `decisions/`
- `technical_docs/`

`dataset_manifest.json` is validation metadata and should not be used as a RAG source.

{'file_path': '/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/data/README.md', 'file_name': 'README.md', 'file_type': 'text/markdown', 'file_size': 298, 'creation_date': '2026-08-21', 'last_modified_date': '2026-08-21'}


In [15]:
for i, doc in enumerate(documents[:10]):
    print(
        i,
        doc.metadata.get("file_name"),
        doc.metadata.get("file_path")
    )

0 README.md /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/data/README.md
1 decision_d001.md /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/data/decisions/decision_d001.md
2 decision_d002.md /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/data/decisions/decision_d002.md
3 decision_d003.md /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/data/decisions/decision_d003.md
4 decision_d004.md /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/data/decisions/decision_d004.md
5 decision_d005.md /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/data/decisions/decision_d005.md
6 decision_d006.md /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/data/decisions/decision_d006.md
7 decision_d007.md /Users/rohitk/mastering_a